# Pong DQN — Google Colab

Train and evaluate a DQN agent on `PongNoFrameskip-v4` using Stable-Baselines3 and rl\_zoo3.
This notebook encapsulates both `train.py` and `evaluate.py` from the repository.

> **Before running:**
> 1. Set runtime to **GPU** (`Runtime → Change runtime type → T4 GPU`).
> 2. Add your W&B API key as a Colab secret named `WANDB_API_KEY` (lock icon in the left panel).
> 3. Update `wandb_entity` in the **Configuration** cell if needed.

**Persistence strategy:** All model checkpoints, TensorBoard logs, and evaluation GIFs are written
directly to Google Drive so nothing is lost if the runtime disconnects.
To resume an interrupted training run, set `resume_model_path` in the Configuration cell.

## 1 — Install dependencies

In [ ]:
!pip install -q stable-baselines3[extra] rl_zoo3 "gymnasium[atari]" ale-py "autorom[accept-rom-license]" wandb imageio imageio-ffmpeg numpy
!AutoROM --accept-license --quiet
print('Dependencies installed.')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('Go to Runtime -> Change runtime type -> T4 GPU')

## 2 — Mount Google Drive

All outputs are written here. Checkpoints are saved every 200k steps,
so at most 200k steps of work are lost on a disconnection.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE  = '/content/drive/MyDrive/pong-DQN'
LOG_DIR     = f'{DRIVE_BASE}/logs'
RESULTS_DIR = f'{DRIVE_BASE}/results'
HYPER_DIR   = '/content/hyperparams'

for d in [LOG_DIR, RESULTS_DIR, HYPER_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Logs     -> {LOG_DIR}')
print(f'Results  -> {RESULTS_DIR}')

## 3 — Weights & Biases login

The notebook reads `WANDB_API_KEY` from Colab secrets automatically.
If the secret is not set, it falls back to an interactive prompt.

In [ ]:
import wandb

try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
    print('Logged in via Colab secret.')
except Exception:
    print('Secret not found – enter your W&B API key manually:')
    wandb.login()

## 4 — Configuration

Edit the values in this cell before starting a run.
The `CONFIG` dict mirrors `config.yml`; `FRAME_STACK` mirrors `hyperparams/dqn_stack_8.yml`.

In [ ]:
# ── Edit these values ────────────────────────────────────────────────────────
CONFIG = {
    'algo':              'dqn',
    'env':               'PongNoFrameskip-v4',
    'device':            'cuda',          # 'cuda' or 'cpu'
    'n_timesteps':       10_000_000,
    'seed':              42,
    'eval_freq':         50_000,
    'eval_episodes':     10,
    'save_freq':         200_000,
    'log_folder':        LOG_DIR,         # writes checkpoints directly to Drive
    'conf_file':         f'{HYPER_DIR}/dqn_stack_8.yml',
    'use_wandb':         True,
    'wandb_project':     'pong-dqn',
    'wandb_entity':      'rogervendrell-universitat-aut-noma-de-barcelona',
    # Set to a .zip checkpoint path to resume training; None = start fresh
    'resume_model_path': None,
}

FRAME_STACK     = 8    # must match frame_stack in the hyperparams file
N_EVAL_EPISODES = 100
GIF_FPS         = 30
# ─────────────────────────────────────────────────────────────────────────────
print('Configuration set.')

In [ ]:
# Write the hyperparameter YAML that rl_zoo3 reads during training.
# Equivalent to hyperparams/dqn_stack_8.yml in the repository.
import yaml

hyper_config = {
    'PongNoFrameskip-v4': {
        'env_wrapper':            ['stable_baselines3.common.atari_wrappers.AtariWrapper'],
        'frame_stack':            FRAME_STACK,
        'policy':                 'CnnPolicy',
        'buffer_size':            16000,
        'exploration_fraction':   0.1,
        'exploration_final_eps':  0.01,
        'learning_starts':        100000,
        'learning_rate':          1e-4,
        'batch_size':             32,
        'train_freq':             4,
        'gradient_steps':         1,
        'target_update_interval': 1000,
        'n_timesteps':            10_000_000,
        'optimize_memory_usage':  False,
    }
}

with open(CONFIG['conf_file'], 'w') as fh:
    yaml.dump(hyper_config, fh, default_flow_style=False)

print('Hyperparams written ->', CONFIG['conf_file'])

---
## Part 1 — Training

Mirrors `train.py`: builds the rl\_zoo3 argument list and delegates to its `train()` function.
Checkpoints land in `MyDrive/pong-DQN/logs/` every 200k steps.

> **Time estimate:** ~8–12 h on a T4 GPU for the full 10M-step run.
> The free Colab tier has session time limits; use `resume_model_path` to continue
> from the last saved checkpoint after reconnecting.

In [ ]:
import sys
from rl_zoo3.train import train

log_folder = CONFIG['log_folder']

sys.argv = [
    'train',
    '--algo',          CONFIG['algo'],
    '--env',           CONFIG['env'],
    '--n-timesteps',   str(CONFIG['n_timesteps']),
    '--seed',          str(CONFIG['seed']),
    '--eval-freq',     str(CONFIG['eval_freq']),
    '--eval-episodes', str(CONFIG['eval_episodes']),
    '--save-freq',     str(CONFIG['save_freq']),
    '--log-folder',    log_folder,
    '--conf-file',     CONFIG['conf_file'],
    '--device',        CONFIG['device'],
    '--tensorboard-log', f'{log_folder}/tensorboard',
    '--progress',
    '--verbose', '1',
]

if CONFIG['use_wandb']:
    sys.argv += ['--track', '--wandb-project-name', CONFIG['wandb_project']]
    if CONFIG.get('wandb_entity'):
        sys.argv += ['--wandb-entity', CONFIG['wandb_entity']]

if CONFIG.get('resume_model_path'):
    sys.argv += ['--trained-agent', CONFIG['resume_model_path']]

train()

---
## Part 2 — Evaluation

Mirrors `evaluate.py`: loads the best saved model, runs `N_EVAL_EPISODES` deterministic
episodes, reports statistics, and exports GIF animations of the best and worst episodes.

Results are saved to `MyDrive/pong-DQN/results/` and logged to W&B.

In [ ]:
# Locate the trained model.
# rl_zoo3 saves to: {log_folder}/{algo}/{env}_{run_id}/best_model.zip
import glob

log_dir = CONFIG['log_folder']

pattern = os.path.join(log_dir, CONFIG['algo'], 'PongNoFrameskip-v4_*', 'best_model.zip')
candidates = glob.glob(pattern)

if not candidates:
    # Fall back to the final model if best_model.zip is absent
    pattern = os.path.join(log_dir, CONFIG['algo'], 'PongNoFrameskip-v4_*', 'PongNoFrameskip-v4.zip')
    candidates = glob.glob(pattern)

if not candidates:
    raise FileNotFoundError(f'No trained model found under {log_dir}. Has training completed?')

MODEL_PATH = max(candidates, key=os.path.getmtime)
print('Using model:', MODEL_PATH)

In [ ]:
# Evaluation helper functions — direct port of evaluate.py
import ale_py
import gymnasium as gym
import imageio
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)


def make_env(n_stack=4, render_mode=None):
    """Build a vectorised PongNoFrameskip-v4 env matching the training setup."""
    env = make_atari_env(
        'PongNoFrameskip-v4',
        n_envs=1,
        env_kwargs={'render_mode': render_mode} if render_mode else {},
    )
    return VecFrameStack(env, n_stack=n_stack)


def run_episode(env, model, record=False):
    """Run one episode; return (total_reward, frames)."""
    obs = env.reset()
    done = False
    total_reward = 0.0
    frames = []

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, _ = env.step(action)
        done = bool(dones[0])
        total_reward += float(rewards[0])

        if record:
            frame = env.render()
            if isinstance(frame, list):
                frame = frame[0]
            if frame is not None:
                frames.append(frame)

    return total_reward, frames


def save_gif(frames, path, fps=30):
    """Save a list of RGB arrays as an animated GIF."""
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    imageio.mimwrite(path, frames, duration=int(1000 / fps), loop=0)
    print(f'  Saved: {path}  ({len(frames)} frames @ {fps} fps)')


print('Evaluation functions defined.')

In [ ]:
print('Loading model:', MODEL_PATH)
model = DQN.load(MODEL_PATH)
print('Model loaded.\n')

env = make_env(n_stack=FRAME_STACK, render_mode='rgb_array')

rewards = []
best_reward,  worst_reward  = -float('inf'),  float('inf')
best_frames,  worst_frames  = [], []

print(f'Running {N_EVAL_EPISODES} evaluation episodes...')
for ep in range(N_EVAL_EPISODES):
    reward, frames = run_episode(env, model, record=True)
    rewards.append(reward)

    if reward > best_reward:
        best_reward = reward
        best_frames = frames
    if reward < worst_reward:
        worst_reward = reward
        worst_frames = frames

    if (ep + 1) % 10 == 0:
        print(f'  Episodes done: {ep + 1}/{N_EVAL_EPISODES}  (last reward: {reward:+.0f})')

env.close()

rewards_arr = np.array(rewards)
print('\n' + '=' * 50)
print('Evaluation Results')
print('=' * 50)
print(f'  Episodes     : {N_EVAL_EPISODES}')
print(f'  Mean reward  : {rewards_arr.mean():+.2f}')
print(f'  Std dev      : {rewards_arr.std():.2f}')
print(f'  Min reward   : {rewards_arr.min():+.0f}')
print(f'  Max reward   : {rewards_arr.max():+.0f}')
print('=' * 50)

In [ ]:
# Save GIFs and summary to Google Drive, then log everything to W&B.
GIF_BEST  = os.path.join(RESULTS_DIR, f'best_episode_reward{best_reward:+.0f}.gif')
GIF_WORST = os.path.join(RESULTS_DIR, f'worst_episode_reward{worst_reward:+.0f}.gif')

print('Saving GIFs to Google Drive...')
save_gif(best_frames,  GIF_BEST,  fps=GIF_FPS)
save_gif(worst_frames, GIF_WORST, fps=GIF_FPS)

summary_path = os.path.join(RESULTS_DIR, 'eval_summary.txt')
with open(summary_path, 'w') as fh:
    fh.write(f'model: {MODEL_PATH}\n')
    fh.write(f'episodes: {N_EVAL_EPISODES}\n')
    fh.write(f'mean_reward: {rewards_arr.mean():+.2f}\n')
    fh.write(f'std_dev: {rewards_arr.std():.2f}\n')
    fh.write(f'min_reward: {rewards_arr.min():+.0f}\n')
    fh.write(f'max_reward: {rewards_arr.max():+.0f}\n')
    fh.write(f'best_gif: {GIF_BEST}\n')
    fh.write(f'worst_gif: {GIF_WORST}\n')
print('Summary saved ->', summary_path)

if CONFIG['use_wandb']:
    eval_run = wandb.init(
        project=CONFIG['wandb_project'],
        entity=CONFIG.get('wandb_entity'),
        job_type='evaluation',
        name='eval',
        reinit=True,
    )
    eval_run.log({
        'eval/mean_reward': float(rewards_arr.mean()),
        'eval/std_reward':  float(rewards_arr.std()),
        'eval/min_reward':  float(rewards_arr.min()),
        'eval/max_reward':  float(rewards_arr.max()),
        'eval/n_episodes':  N_EVAL_EPISODES,
        'best_episode':     wandb.Video(GIF_BEST,  format='gif'),
        'worst_episode':    wandb.Video(GIF_WORST, format='gif'),
    })
    eval_run.finish()
    print('Results logged to W&B.')

In [ ]:
# Display GIFs inline.
from IPython.display import Image, display

print('Best episode (reward', best_reward, '):')
display(Image(filename=GIF_BEST))

print('Worst episode (reward', worst_reward, '):')
display(Image(filename=GIF_WORST))